# 00 — LangChain Deep Agents Quickstart

**来源:** [LangChain Docs — Deep Agents Quickstart (Python)](https://docs.langchain.com/oss/python/deepagents/quickstart)

本指南带你从零搭建第一个 Deep Agent — 一个具备**规划、搜索、文件系统、子 Agent 委派**能力的研究型 Agent。
### 核心自动能力
1. **规划** — 内置 `write_todos` 工具拆解任务
2. **调研** — 调用搜索工具收集信息
3. **上下文管理** — 通过文件系统工具 (`write_file`, `read_file`) 卸载大量内容
4. **子 Agent 委派** — 按需将复杂子任务委托给隔离的子 Agent
5. **综合** — 将发现编译为连贯的回复

## 1. 前置条件

需要一个支持 **tool calling** 的 LLM API Key。

支持的 Provider：
- Google Gemini
- OpenAI
- Anthropic
- OpenRouter
- Fireworks
- Baseten
- Ollama（本地）
- 及其他 [LangChain Chat Model](https://docs.langchain.com/oss/python/integrations/chat)

## 2. 安装依赖

In [ ]:
pip install deepagents tavily-python

## 3. 设置 API Key

根据你的模型提供商设置环境变量。以 DeepSeek 为例：

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# ---- 模型 API Key（选你的提供商） ----
# os.environ["GOOGLE_API_KEY"] = "..."
# os.environ["OPENAI_API_KEY"] = "..."
# os.environ["ANTHROPIC_API_KEY"] = "..."
# os.environ["OPENROUTER_API_KEY"] = "..."
# os.environ["FIREWORKS_API_KEY"] = "..."
# os.environ["BASETEN_API_KEY"] = "..."

# ---- 搜索 API Key ----
# 本教程使用 Tavily 作为搜索提供商，可替换为 DuckDuckGo / SerpAPI / Brave Search
# os.environ["TAVILY_API_KEY"] = "tvly-..."

print("API keys configured (placeholder).")

## 4. 创建搜索工具

Tavily 搜索封装为 Deep Agents 可调用的 tool：

In [ ]:
from typing import Literal
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )


print("Search tool created.")

## 5. 创建 Deep Agent

通过 `provider:model` 字符串传入模型，或传入预初始化的 Model 实例。

In [ ]:
from deepagents import create_deep_agent

# 系统提示词：引导 Agent 成为专家研究员
research_instructions = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

# 创建 Agent（传入模型字符串，自动初始化）
agent = create_deep_agent(
    model="deepseek-v4-flash",  # 替换为你的模型
    tools=[internet_search],
    system_prompt=research_instructions,
)

print("Deep Agent created successfully.")

### 5.1 支持的模型格式

`create_deep_agent(model=...)` 支持两种方式：

| 方式 | 示例 | 说明 |
|------|------|------|
| **字符串** | `"google_genai:gemini-3.5-flash"` | 传入 `provider:model`，自动初始化 |
| **实例** | `init_chat_model("deepseek-v4-flash")` | 预先配置好参数再传入 |

完整 Provider 列表参考：[Chat Model Integrations](https://docs.langchain.com/oss/python/integrations/chat)

## 6. 运行 Agent

In [ ]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is LangGraph?"}
    ]
})

# 打印 Agent 的最终回复
print(result["messages"][-1].content)

## 7. 流式输出

Deep Agents 内置流式支持，基于 LangGraph 实现实时更新：

In [ ]:
for chunk in agent.stream({
    "messages": [
        {"role": "user", "content": "Explain LangChain in simple terms."}
    ]
}):
    if "agent" in chunk:
        print(chunk["agent"]["messages"][-1].content, end="", flush=True)

## 8. 追踪（LangSmith）

通过 LangSmith 追踪 Agent 的每一步：
- 规划步骤
- 工具调用
- 子 Agent 委派

设置方式：

In [ ]:
import os
os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = "lsv2_..."

print("LangSmith tracing enabled (placeholder).")

## 9. 下一步

| 主题 | 文档 | 说明 |
|------|------|------|
| **自定义 Agent** | [Customization](https://docs.langchain.com/oss/python/deepagents/customization) | 自定义 system prompt、tools、subagents |
| **长期记忆** | [Memory](https://docs.langchain.com/oss/python/deepagents/memory) | 跨会话持久化记忆 |
| **生产部署** | [Managed Deep Agents](https://docs.langchain.com/langsmith/deploy-managed-deep-agent) | 在 LangSmith 中创建、运行、运维 |
| **更多示例** | [Examples](https://docs.langchain.com/oss/python/deepagents/examples) | 各种应用场景的完整示例 |
| **模型参考** | [Models](https://docs.langchain.com/oss/python/deepagents/models) | 全部支持的 Provider 与评测数据 |
| **Context Engineering** | [Context Engineering](https://docs.langchain.com/oss/python/deepagents/context-engineering) | 五种上下文类型详解 |

---
**参考链接**

- [Deep Agents Quickstart — LangChain Docs](https://docs.langchain.com/oss/python/deepagents/quickstart)
- [Deep Agents Overview](https://docs.langchain.com/oss/python/deepagents/overview)
- [Documentation Index](https://docs.langchain.com/llms.txt)